# EasyGen Pro — Image-to-Image Backend
**Runtime → Change runtime type → T4 GPU** before running!

Run cells **1 → 5** in order. After Cell 5 you will get an ngrok URL — paste it into `app.html` as `BASE_URL_GEMINI`.

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────
# diffusers>=0.30 fixes the 'diffusers.guiders' ModuleNotFoundError
!pip install -q --upgrade diffusers transformers accelerate
!pip install -q xformers flask flask-cors pyngrok pillow

# Must restart runtime after upgrading diffusers/transformers
print('✅ Packages installed — RESTARTING RUNTIME NOW …')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.8 MB/s eta 0:00:00
✅ Packages installed — RESTARTING RUNTIME NOW …


In [ ]:
# ── CELL 2: Set ngrok auth token ───────────────────────────────────────
# Get your FREE token at https://ngrok.com → Dashboard → Your Authtoken
from pyngrok import ngrok

NGROK_TOKEN = "3AAR2EklwbVY38BPNlJZQf9lKLr_7ULgVjE6Eak9U5y3cnmhb"   # ← replace this

ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token set')

✅ ngrok token set


In [ ]:
# ── CELL 3: Load SD img2img pipeline ──
import torch
from diffusers import (
    StableDiffusionImg2ImgPipeline,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    PNDMScheduler,
    UniPCMultistepScheduler,
)
from PIL import Image

# ✅ No token needed — publicly accessible, same quality as SD 2.1
MODEL_ID = "SG161222/Realistic_Vision_V5.1_noVAE"

print(f"⏳ Loading {MODEL_ID} — downloads ~4 GB on first run …")

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe = pipe.to("cuda")

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("  xformers attention enabled")
except Exception:
    pipe.enable_attention_slicing(1)
    print("  attention slicing enabled")

pipe.enable_vae_tiling()

print(f"✅ Pipeline ready on {torch.cuda.get_device_name(0)}")

⏳ Loading SG161222/Realistic_Vision_V5.1_noVAE — downloads ~4 GB on first run …


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--SG161222--Realistic_Vision_V5.1_noVAE/snapshots/1e9f017a7b1eaefb63a1900ea6c5953d2739fd21/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  xformers attention enabled
✅ Pipeline ready on Tesla T4


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2290: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


In [ ]:
# ── CELL 4: Scheduler helper ────────────────────────────────────────────
SCHEDULER_MAP = {
    "UniPC":              UniPCMultistepScheduler,
    "DPMSolverMultistep": DPMSolverMultistepScheduler,
    "DDIM":               DDIMScheduler,
    "EulerAncestral":     EulerAncestralDiscreteScheduler,
    "PNDM":               PNDMScheduler,
}

def set_scheduler(name: str):
    cls = SCHEDULER_MAP.get(name, UniPCMultistepScheduler)
    # Fix: use_karras_sigmas=False prevents the final_sigmas_type conflict
    try:
        pipe.scheduler = cls.from_config(
            pipe.scheduler.config,
            use_karras_sigmas=False,
            algorithm_type="dpmsolver++"
        )
    except TypeError:
        # Some schedulers don't accept these args — fall back cleanly
        pipe.scheduler = cls.from_config(pipe.scheduler.config)
    print(f"  Scheduler → {cls.__name__}")

print("✅ Scheduler helper ready")

✅ Scheduler helper ready


In [ ]:
# ── CELL 5: Flask server + ngrok tunnel (KEEP THIS CELL RUNNING) ────────
import io, threading
from flask import Flask, request, send_file, jsonify
from flask_cors import CORS
from PIL import Image
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_ID})


@app.route("/api/img2img", methods=["POST"])
def img2img():
    try:
        # ── Validate ────────────────────────────────────────────────
        if "image" not in request.files:
            return jsonify({"error": "No image file uploaded"}), 400

        prompt = request.form.get("prompt", "").strip()
        if not prompt:
            return jsonify({"error": "Prompt is required"}), 400

        # ── Parse params ─────────────────────────────────────────────
        neg_prompt     = request.form.get("neg_prompt",
                           "blurry, low quality, distorted, deformed, "
                           "watermark, text, bad anatomy, duplicate, ugly")
        strength       = float(request.form.get("strength",       0.6))
        guidance_scale = float(request.form.get("guidance_scale", 7.5))
        num_steps      = int(request.form.get("num_steps",        40))
        scheduler_name = request.form.get("scheduler",            "UniPC")
        out_w          = int(request.form.get("width",            768))
        out_h          = int(request.form.get("height",           768))

        # Clamp to safe ranges
        strength       = max(0.1, min(1.0, strength))
        guidance_scale = max(1.0, min(20.0, guidance_scale))
        num_steps      = max(10, min(80, num_steps))
        out_w          = min(out_w, 1024)
        out_h          = min(out_h, 1024)

        print(f"\n🎨 New request")
        print(f"   prompt    : {prompt[:80]}")
        print(f"   strength  : {strength}  cfg: {guidance_scale}  steps: {num_steps}")
        print(f"   scheduler : {scheduler_name}  size: {out_w}x{out_h}")

        # ── Preprocess image ─────────────────────────────────────────
        raw = Image.open(io.BytesIO(request.files["image"].read())).convert("RGB")
        tgt_w = (out_w // 8) * 8
        tgt_h = (out_h // 8) * 8
        init_image = raw.resize((tgt_w, tgt_h), Image.LANCZOS)

        # ── Set scheduler ────────────────────────────────────────────
        set_scheduler(scheduler_name)

        # ── Run pipeline ─────────────────────────────────────────────
        with torch.autocast("cuda"):
            result = pipe(
                prompt              = prompt,
                negative_prompt     = neg_prompt,
                image               = init_image,
                strength            = strength,
                guidance_scale      = guidance_scale,
                num_inference_steps = num_steps,
            )

        # ── Return PNG ───────────────────────────────────────────────
        buf = io.BytesIO()
        result.images[0].save(buf, format="PNG")
        buf.seek(0)
        print("✅ Done")
        return send_file(buf, mimetype="image/png")

    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({"error": str(e)}), 500


# ── Start Flask in background thread ────────────────────────────────
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()

# ── Open ngrok tunnel ────────────────────────────────────────────────
public_url = ngrok.connect(5000).public_url
print("\n" + "="*60)
print("🚀  Backend is LIVE at:")
print(f"    {public_url}")
print("="*60)
print("\n👉  Copy the URL above into app.html:")
print(f'    const BASE_URL_GEMINI = "{public_url}";')

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



🚀  Backend is LIVE at:
    https://sanious-potentially-journey.ngrok-free.dev

👉  Copy the URL above into app.html:
    const BASE_URL_GEMINI = "https://sanious-potentially-journey.ngrok-free.dev";


In [ ]:
# ── CELL 6 (Optional): Quick test without a browser ─────────────────────
# Upload any image to Colab as /content/test_input.jpg first, then run this.

import requests
from IPython.display import display, Image as IPImage

# Use the public_url from Cell 5
TEST_URL = str(public_url)

with open("/content/test_input.jpg", "rb") as f:
    r = requests.post(
        TEST_URL + "/api/img2img",
        files={"image": ("test.jpg", f, "image/jpeg")},
        data={
            "prompt":         "cyberpunk neon city style, ultra detailed",
            "neg_prompt":     "blurry, low quality",
            "strength":       "0.65",
            "guidance_scale": "7.5",
            "num_steps":      "40",
            "scheduler":      "DPMSolverMultistep",
            "width":          "768",
            "height":         "768",
        }
    )

if r.status_code == 200:
    with open("/content/output.png", "wb") as out:
        out.write(r.content)
    display(IPImage("/content/output.png"))
    print("✅ Test passed — output saved to /content/output.png")
else:
    print("❌ Error:", r.status_code, r.text)

FileNotFoundError: [Errno 2] No such file or directory: '/content/test_input.jpg'